# Similarity with other embedding models

Same pipeline as `3_similarity.ipynb` / `3b_similarity_expanded.ipynb`, but the semantic signal comes from a configurable embedding model. TF-IDF, fuzzy matching, weights, calibration and the winter/windproof penalty are unchanged, so only the embedding model differs between the matrices. **To switch model, edit `RUN` in the first code cell.**

Output: `data/similarity_<tag>.npz` (original feature names) and `data/similarity_expanded_<tag>.npz` (expanded catalog). The existing `similarity.npz` and `similarity_expanded.npz` are the `qwen3_0_6b` baseline.

The last cell clusters every matrix with DBSCAN under the rule from `6_dbscan.ipynb` (noise as one-point clusters, coverage ≥ `MIN_COVERAGE`, best score) so the models can be compared.

In [1]:
import json
import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import requests
import torch
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25  # same weights as 3_similarity
W_NAME = 0.50  # weight of the feature-name vector against the description vector
PENALTY = 0.30
QWEN_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

# ---- choose what to run: edit RUN ---------------------------------------------------
RUN = ["gemini_embedding_2", "voyage_4_large", "cohere_embed_5_pro", "nemotron_8b", "qwen3_8b_api", "openai_3_large"]  # tags from MODELS below; only these are loaded or called
OVERWRITE = False  # False: skip a tag whose .npz files already exist

# tag -> (backend, model id, query prompt or None)
#   "local":      Hugging Face model run on this machine (first use downloads it)
#   "openrouter": hosted model, needs OPENROUTER_API_KEY in the environment (a few cents)
MODELS = {
    "qwen3_4b": ("local", "Qwen/Qwen3-Embedding-4B", QWEN_PROMPT),
    "qwen3_8b": ("local", "Qwen/Qwen3-Embedding-8B", QWEN_PROMPT),
    "bge_m3": ("local", "BAAI/bge-m3", None),
    "gemini_embedding_2": ("openrouter", "google/gemini-embedding-2", None),
    "voyage_4_large": ("openrouter", "mongodb/voyage-4-large", None),
    "cohere_embed_5_pro": ("openrouter", "cohere/embed-5-pro", None),
    "nemotron_8b": ("openrouter", "nvidia/nemotron-3.5-embed-8b:free", None),
    "qwen3_8b_api": ("openrouter", "qwen/qwen3-embedding-8b", QWEN_PROMPT),
    "openai_3_large": ("openrouter", "openai/text-embedding-3-large", None),
}
# output prefix -> features file
DATASETS = {
    "similarity": "data/features_en.json",
    "similarity_expanded": "data/features_en_expanded.json",
}
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.bfloat16  # 8B in float32 would not fit in 36 GB

W1128 02:01:35.268000 77421 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Model-independent signals (per dataset)

TF-IDF, fuzzy and the penalty do not depend on the embedding model, so they are computed once per dataset.

In [2]:
CONCEPTS = {"winter": r"\b(winter|cold|freez\w*|frost\w*)\b", "windproof": r"\b(wind\w*)\b"}
CONFLICTS = [("winter", "windproof")]


def upper(matrix):
    return matrix[np.triu_indices(len(matrix), k=1)]


def load_dataset(path):
    data = json.loads(Path(path).read_text(encoding="utf8"))
    records = data["attribute_records"]
    names = [r["title"] for r in records]
    descriptions = [r["descriptions"] for r in records]

    documents = [" ".join(texts) for texts in descriptions]
    tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
    tfidf.fit([d for d in documents if d] + data["unpaired_descriptions"])
    vectors = tfidf.transform(documents)
    tfidf_sim = (vectors @ vectors.T).toarray()

    fuzzy = process.cdist(names, names, scorer=fuzz.token_sort_ratio, processor=utils.default_process) / 100
    base = np.median(upper(fuzzy))
    fuzzy = np.clip((fuzzy - base) / (1 - base), 0, 1)

    concepts = [{c for c, p in CONCEPTS.items() if re.search(p, n, re.IGNORECASE)} for n in names]
    penalty = np.zeros_like(tfidf_sim)
    for a, b in CONFLICTS:
        only_a = np.array([a in c and b not in c for c in concepts])
        only_b = np.array([b in c and a not in c for c in concepts])
        penalty[np.ix_(only_a, only_b)] = PENALTY
        penalty[np.ix_(only_b, only_a)] = PENALTY

    return {"names": names, "descriptions": descriptions, "tfidf": tfidf_sim, "fuzzy": fuzzy, "penalty": penalty}


datasets = {prefix: load_dataset(path) for prefix, path in DATASETS.items()}
for prefix, d in datasets.items():
    print(prefix, len(d["names"]), "feature names")

similarity 445 feature names
similarity_expanded 664 feature names


## 2. Embed and blend

For each feature name the model embeds the name and its distinct descriptions; the description vectors are averaged by frequency and blended with the name vector (`W_NAME`). Cosine similarities are calibrated so that the median pair is 0, then blended with TF-IDF and fuzzy exactly as in `3_similarity.ipynb`.

In [3]:
def local_encoder(model_id, prompt):
    model = SentenceTransformer(model_id, device=DEVICE, model_kwargs={"dtype": DTYPE})
    return lambda texts: model.encode(texts, prompt=prompt, normalize_embeddings=True, show_progress_bar=False, batch_size=16)


def openrouter_encoder(model_id, prompt):
    headers = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}

    def encode(texts, batch=64):
        out = []
        for i in range(0, len(texts), batch):
            r = requests.post("https://openrouter.ai/api/v1/embeddings", headers=headers, timeout=120,
                              json={"model": model_id, "input": [(prompt or "") + t for t in texts[i:i + batch]]})
            r.raise_for_status()
            out += [item["embedding"] for item in sorted(r.json()["data"], key=lambda item: item["index"])]
        v = np.asarray(out, dtype=np.float32)
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    return encode


def semantic_similarity(encode, names, descriptions):
    name_vectors = encode(names)
    all_texts = sorted({t for texts in descriptions for t in texts})
    text_vector = dict(zip(all_texts, encode(all_texts)))

    vectors = []
    for name_vector, texts in zip(name_vectors, descriptions):
        if texts:
            counts = Counter(texts)
            mean = np.average([text_vector[t] for t in counts], axis=0, weights=list(counts.values()))
            v = W_NAME * name_vector + (1 - W_NAME) * mean
            v = v / np.linalg.norm(v)
        else:
            v = name_vector
        vectors.append(v)
    vectors = np.asarray(vectors, dtype=np.float32)
    assert np.isfinite(vectors).all(), "embedding produced NaN/inf, try another DTYPE"

    raw = vectors @ vectors.T
    base = np.median(upper(raw))
    return np.clip((raw - base) / (1 - base), 0, 1)


for tag in RUN:
    backend, model_id, prompt = MODELS[tag]
    outs = {prefix: Path(f"data/{prefix}_{tag}.npz") for prefix in datasets}
    if not OVERWRITE and all(p.is_file() for p in outs.values()):
        print(f"{tag}: files exist, skipped (OVERWRITE = True to recompute)")
        continue
    encode = local_encoder(model_id, prompt) if backend == "local" else openrouter_encoder(model_id, prompt)  # loaded only now
    for prefix, d in datasets.items():
        semantic = semantic_similarity(encode, d["names"], d["descriptions"])
        similarity = W_SEMANTIC * semantic + W_TFIDF * d["tfidf"] + W_FUZZY * d["fuzzy"]
        np.fill_diagonal(similarity, 1.0)
        similarity = np.clip(similarity - d["penalty"], 0, 1)
        np.savez_compressed(outs[prefix], similarity=similarity, titles=np.array(d["names"]))
        print(f"{outs[prefix]}: {similarity.shape}, median {np.median(upper(similarity)):.3f}, pairs >= 0.5: {(upper(similarity) >= 0.5).mean():.2%}")
    del encode
    if DEVICE == "mps":
        torch.mps.empty_cache()

gemini_embedding_2: files exist, skipped (OVERWRITE = True to recompute)


data/similarity_voyage_4_large.npz: (445, 445), median 0.027, pairs >= 0.5: 0.26%


data/similarity_expanded_voyage_4_large.npz: (664, 664), median 0.027, pairs >= 0.5: 0.21%


data/similarity_cohere_embed_5_pro.npz: (445, 445), median 0.028, pairs >= 0.5: 0.23%


data/similarity_expanded_cohere_embed_5_pro.npz: (664, 664), median 0.028, pairs >= 0.5: 0.19%


data/similarity_nemotron_8b.npz: (445, 445), median 0.020, pairs >= 0.5: 0.19%


data/similarity_expanded_nemotron_8b.npz: (664, 664), median 0.021, pairs >= 0.5: 0.14%


data/similarity_qwen3_8b_api.npz: (445, 445), median 0.027, pairs >= 0.5: 0.37%


data/similarity_expanded_qwen3_8b_api.npz: (664, 664), median 0.027, pairs >= 0.5: 0.30%


data/similarity_openai_3_large.npz: (445, 445), median 0.024, pairs >= 0.5: 0.23%


data/similarity_expanded_openai_3_large.npz: (664, 664), median 0.024, pairs >= 0.5: 0.19%


## 3. Compare with DBSCAN

Each matrix is clustered over the grid of `6_dbscan.ipynb`. A candidate needs at least 2 clusters and coverage ≥ `MIN_COVERAGE`; the score is the silhouette with every noise point as its own one-point cluster. The best candidate per matrix is listed. Silhouette (grouped) is over the clustered feature names only.

In [4]:
MIN_COVERAGE = 0.50
EPS_VALUES = np.round(np.arange(0.20, 0.82, 0.02), 2)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]


def best_dbscan(similarity):
    D = 1 - similarity
    n = len(D)
    best = None
    for eps in EPS_VALUES:
        for m in MIN_SAMPLES_VALUES:
            labels = DBSCAN(eps=eps, min_samples=m, metric="precomputed").fit_predict(D)
            clustered = labels != -1
            coverage = clustered.mean()
            k = len(set(labels[clustered]))
            if k < 2 or coverage < MIN_COVERAGE:
                continue
            relabeled = labels.copy()
            relabeled[~clustered] = np.arange(k, k + (~clustered).sum())  # noise = one-point clusters
            score = silhouette_score(D, relabeled, metric="precomputed")
            key = (score, coverage, -k)
            if best is None or key > best[0]:
                grouped = silhouette_score(D[np.ix_(clustered, clustered)], labels[clustered], metric="precomputed")
                best = (key, eps, m, k, coverage, grouped)
    return best


rows = []
for prefix in DATASETS:
    for tag in ["qwen3_0_6b"] + list(MODELS):  # every matrix found on disk is compared
        path = Path(f"data/{prefix}.npz" if tag == "qwen3_0_6b" else f"data/{prefix}_{tag}.npz")
        if not path.is_file():
            continue
        best = best_dbscan(np.load(path)["similarity"])
        if best is None:
            rows.append((prefix, tag, "no candidate passes the filters"))
            continue
        (score, *_), eps, m, k, coverage, grouped = best
        rows.append((prefix, tag, f"score {score:.4f}  eps {eps:.2f}  m {m}  clusters {k}  coverage {coverage:.1%}  silhouette (grouped) {grouped:.4f}"))

for prefix, tag, text in rows:
    print(f"{prefix:<20}{tag:<20}{text}")

similarity          qwen3_0_6b          score 0.1163  eps 0.50  m 2  clusters 60  coverage 56.2%  silhouette (grouped) 0.3027
similarity          gemini_embedding_2  score 0.1127  eps 0.50  m 2  clusters 60  coverage 50.3%  silhouette (grouped) 0.3286
similarity          voyage_4_large      score 0.1272  eps 0.48  m 2  clusters 63  coverage 50.3%  silhouette (grouped) 0.3498
similarity          cohere_embed_5_pro  score 0.0931  eps 0.52  m 2  clusters 56  coverage 53.9%  silhouette (grouped) 0.2857
similarity          nemotron_8b         score 0.1235  eps 0.52  m 2  clusters 67  coverage 51.9%  silhouette (grouped) 0.3356
similarity          qwen3_8b_api        score 0.1365  eps 0.46  m 2  clusters 64  coverage 53.0%  silhouette (grouped) 0.3589
similarity          openai_3_large      score 0.1307  eps 0.50  m 2  clusters 65  coverage 52.8%  silhouette (grouped) 0.3416
similarity_expanded qwen3_0_6b          score 0.0779  eps 0.48  m 2  clusters 75  coverage 51.8%  silhouette (grouped)